In this notebook we compare the networkx library with our library. 

It is important to note that conventions differs:
* network_simplex: ``outgoing - incoming = balance``; supply is positive.
* NetworkX: ``incoming - outgoing = demand``; demand is positive.

Therefore each node is passed to NetworkX with ``demand=-balance.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from network_simplex import Arc, Network, NetworkSimplex, Node

# Comparing a Unique Solution

## Defining the network

In [ ]:
# This instance has a unique optimum, so both the total cost and individual
# arc flows can be compared directly.

NODES = [
    Node("factory", balance=8),
    Node("crossdock", balance=0),
    Node("retail", balance=-5),
    Node("hospital", balance=-3),
]

ARCS = [
    Arc("factory_crossdock", "factory", "crossdock", cost=1),
    Arc("crossdock_retail", "crossdock", "retail", cost=2),
    Arc("crossdock_hospital", "crossdock", "hospital", cost=1),
    Arc("factory_retail", "factory", "retail", cost=6),
    Arc("factory_hospital", "factory", "hospital", cost=5),
]

# Dibujar
plt.figure(figsize=(4, 4))
pos = {
    'factory': (0, 0),
    'crossdock': (3, 1),
    'retail': (4, -1),
    'hospital': (5, 1)
}
# Dibujar nodos
nx.draw_networkx_nodes(graph, pos, node_size=1500, node_color='lightblue')

# Dibujar etiquetas de nodos con balances
labels = {}
for node in NODES:
    labels[node.id] = f"{node.id}\n(b={node.balance})"
nx.draw_networkx_labels(graph, pos, labels, font_size=10)

# Dibujar aristas
nx.draw_networkx_edges(graph, pos, edge_color='gray', arrows=True, arrowsize=50,arrowstyle='->')

# Dibujar etiquetas de aristas (costos)
edge_labels = nx.get_edge_attributes(graph, 'weight')
edge_labels = {k: f"c={v}" for k, v in edge_labels.items()}
nx.draw_networkx_edge_labels(graph, pos, edge_labels, font_size=9)

plt.title("Red de Flujo - Nodos y Aristas")
plt.axis('off')
plt.tight_layout()
plt.show()

## Solving using NetworkX

In [45]:
# Sol
graph = nx.DiGraph()
for node in NODES:
    graph.add_node(node.id, demand=-node.balance)
for arc in ARCS:
    graph.add_edge(arc.tail, arc.head, weight=arc.cost, arc_id=arc.id)

cost_networkx, flow_dict_networkx = nx.network_simplex(graph)
flows_networkx = {
    arc.id: float(flow_dict_networkx[arc.tail][arc.head])
    for arc in ARCS
}

## Solving using our implementation

In [46]:
network = Network(NODES, ARCS)
ours = NetworkSimplex(network).solve()

In [47]:
print("Minimum-cost flow comparison")
print("=" * 31)
print(f"Our Network Simplex cost : {ours.objective_value:g}")
print(f"NetworkX cost            : {cost_networkx:g}")
print(f"Our iterations            : {ours.iterations}")
print()
print(f"{'arc':<24} {'ours':>10} {'networkx':>10}")
print("-" * 48)
for arc_id, our_flow in ours.flows.items():
    print(f"{arc_id:<24} {our_flow:>10g} {flows_networkx[arc_id]:>10g}")


Minimum-cost flow comparison
Our Network Simplex cost : 21
NetworkX cost            : 21
Our iterations            : 3

arc                            ours   networkx
------------------------------------------------
factory_crossdock                 8          8
factory_hospital                  0          0
crossdock_hospital                3          3
crossdock_retail                  5          5
factory_retail                    0          0
